In [5]:
import pandas as pd
import csv

In [6]:
# Load data from git.csv
df = pd.read_csv('git.csv', sep='|', quoting=csv.QUOTE_MINIMAL, escapechar='\\')
df.head()

,Unnamed: 0,project,file,date_start,date_end,library,about
0,0,alice,server.py,2020-06-02 17:02:29,2020-06-02 20:24:54,asyncio,SImple Alice skill python server
1,1,alice,server.py,2020-06-02 17:02:29,2020-06-02 20:24:54,ssl,SImple Alice skill python server
2,2,alice,server.py,2020-06-02 17:02:29,2020-06-02 20:24:54,firebase_admin,SImple Alice skill python server
3,3,alice,server.py,2020-06-02 17:02:29,2020-06-02 20:24:54,aiohttp,SImple Alice skill python server
4,4,alice,server.py,2020-06-02 17:02:29,2020-06-02 20:24:54,firebase_admin,SImple Alice skill python server


In [7]:
# drop all columns except of library
df = pd.DataFrame(df['library'])
df.head()

,library
0,asyncio
1,ssl
2,firebase_admin
3,aiohttp
4,firebase_admin


In [8]:
# Get the unique values of the library column
unique_values = df['library'].unique()
len(unique_values)

645

In [9]:
# Save the unique values to a file: unique_libraries.txt
with open('unique_libraries.txt', 'w') as f:
    for item in unique_values:
        f.write("%s\n" % item)

In [10]:
# Convert each item to text
unique_values = [str(x) for x in unique_values]

#### GPT: Get list of categories

In [11]:
# Ask user the OpenAI token: https://platform.openai.com/account/api-keys
openai_key = input("Enter OpenAI api-key: ")

In [12]:
from openai import OpenAI

def llm_request(model, api_key, messages):
    client = OpenAI(api_key=api_key)

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

In [13]:
import re
import json

def extract_json_from_llm_answer(llm_answer):
    # Find the JSON structure using regex
    json_match = re.search(r'```json\n([\s\S]*?)\n```', llm_answer)
    
    if json_match:
        json_str = json_match.group(1)
        try:
            # Parse the extracted JSON string
            json_data = json.loads(json_str)
            return json_data
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            return None
    else:
        print("No JSON structure found in the LLM answer.")
        return None

In [14]:
system_content = "You are a Python library expert."
cat_count = 24
user_content = f"""There is a list of libraries. You need to determine about {cat_count} categories, that these libraries belong to in general.
"""
# Concatenate a list of libraries as a string with spaces
user_content = " ".join(unique_values)
user_content += f"""
Left one category for "other" category. Provide your list of categories in JSON format. For example:
["Natural language processing", "Data Visualization", "Computer vision", "Speech recognition", "Speech synthesis", "CUDA development", "Distributed Computing", "Others"]
and so on. {cat_count} categories in total. Don't represent the libraries. I need only a list of catefgores."""
messages=[
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content}
    ]

In [15]:
###  Pricing:
# [0.5:0.25] gpt-3.5-turbo-0125
# [1:2] gpt-3.5-turbo-1106
# [1.5:2] gpt-3.5-turbo-0613
# [5:2.5] gpt-4o
model = "gpt-4o"

In [16]:
engine = "openai"
answer = llm_request(model, openai_key, messages)
answer

'```json\n[\n    "Natural Language Processing",\n    "Data Visualization",\n    "Computer Vision",\n    "Speech Recognition",\n    "Speech Synthesis",\n    "CUDA Development",\n    "Distributed Computing",\n    "Web Development",\n    "API Development",\n    "Database Management",\n    "Analytics",\n    "Machine Learning",\n    "Deep Learning",\n    "Reinforcement Learning",\n    "Data Processing",\n    "Cryptography",\n    "Networking",\n    "Command Line Utilities",\n    "Web Scraping",\n    "Testing",\n    "Virtualization and Containers",\n    "Optimization",\n    "Automation",\n    "Others"\n]\n```'

In [17]:
categories = extract_json_from_llm_answer(answer)
# # Get list of keys
# categories = list(extracted_json.keys())
print(len(categories))
categories

24


['Natural Language Processing',
 'Data Visualization',
 'Computer Vision',
 'Speech Recognition',
 'Speech Synthesis',
 'CUDA Development',
 'Distributed Computing',
 'Web Development',
 'API Development',
 'Database Management',
 'Analytics',
 'Machine Learning',
 'Deep Learning',
 'Reinforcement Learning',
 'Data Processing',
 'Cryptography',
 'Networking',
 'Command Line Utilities',
 'Web Scraping',
 'Testing',
 'Virtualization and Containers',
 'Optimization',
 'Automation',
 'Others']

#### GPT: determine the category of each library

In [20]:
###  Pricing:
# [0.5:0.25] gpt-3.5-turbo-0125
# [1:2] gpt-3.5-turbo-1106
# [1.5:2] gpt-3.5-turbo-0613
# [5:2.5] gpt-4o

model = "gpt-3.5-turbo-0125"
print("Categories count:", len(unique_values))
system_content = "You are a Python library expert."
current_category = 0
library_dict = {}
for library in unique_values:
    user_content = f"""Categorize the Python library "{library}" into exactly one of these categories:
{categories}

Respond with ONLY the category name, nothing else. If unsure, choose the closest match."""
    messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content}
        ]
    answer = llm_request(model, openai_key, messages)
    print(f"[{current_category} / {len(unique_values)}] Library: {library}, Category: {answer}")
    library_dict[library] = answer.replace("'", "")
    current_category += 1
print(f"library_dict length: {len(library_dict)}")

Categories count: 645
[0 / 645] Library: asyncio, Category: Distributed Computing
[1 / 645] Library: ssl, Category: Cryptography
[2 / 645] Library: firebase_admin, Category: Database Management
[3 / 645] Library: aiohttp, Category: Web Development
[4 / 645] Library: json, Category: Data Processing
[5 / 645] Library: logging, Category: Others
[6 / 645] Library: pymysql, Category: Database Management
[7 / 645] Library: pandas, Category: Data Processing
[8 / 645] Library: time, Category: Data Processing
[9 / 645] Library: random, Category: Others
[10 / 645] Library: string, Category: Data Processing
[11 / 645] Library: difflib, Category: Others
[12 / 645] Library: __future__, Category: Others
[13 / 645] Library: flask, Category: Web Development
[14 / 645] Library: tongue_twister, Category: Others
[15 / 645] Library: requests, Category: Web Development
[16 / 645] Library: urllib, Category: Web Development
[17 / 645] Library: datetime, Category: Data Processing
[18 / 645] Library: os, Categ

In [22]:
list(library_dict.values())[-10:]

['Automation',
 'Machine Learning',
 'Data Processing',
 'API Development',
 'API Development',
 'Testing',
 'Data Processing',
 'Text Processing',
 'Web Scraping',
 'Data Processing']

In [23]:
print(f"library_dict.values set length: {len(set(library_dict.values()))}")
# Iterate in the ilbrary keys and replace value by 'Others' if it is not in the categories list
for key in library_dict.keys():
    if library_dict[key] not in categories:
        library_dict[key] = "Others"
print(f"library_dict.values set length: {len(set(library_dict.values()))}")

library_dict.values set length: 42
library_dict.values set length: 23


In [24]:
# Save json to cat.json
with open('cat.json', 'w') as f:
    json.dump(library_dict, f)